# 📘 Notebook 17: Summarisation

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module F: Language Models in Practice · Notebook 3 of 4 in this module · (17 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Distinguish **extractive** from **abstractive** summarisation and state the strengths and risks of each
- Build an extractive summariser from **word frequencies**
- Implement **TextRank**: a sentence graph ranked with the PageRank algorithm, written by hand
- Evaluate summaries with **ROUGE-1**, **ROUGE-2** and **ROUGE-L**, computed by hand
- Compare several methods against a baseline on the same text
- Produce an abstractive summary with a pretrained model, and check it for **unsupported claims**

> **Prerequisites:** Notebooks 03, 08 and 13 of this course. Pretrained Transformer models are introduced in *From Python to Deep Learning & Fine-Tuning*, Notebook 17.
>
> 🔭 **Why this notebook matters:** there is far more text than anyone can read. Summarisation, reducing a text to what matters, is among the most requested things a language system can do, and among the most delicate. A summary that leaves out the key point, or worse, states something the source never said, can do real damage in a medical, legal or news setting. This notebook builds summarisers that cannot invent anything, then meets one that can, and shows how to measure both.

> ℹ️ **Setup note.** Run this cell once. Everything except Section 6 runs offline in a few seconds. Section 6 downloads a pretrained summarisation model of about 1.2 GB and is meant for Colab.

In [ ]:
import nltk
for package in ["punkt", "punkt_tab", "stopwords", "reuters"]:
    nltk.download(package, quiet=True)

import re
import random
from collections import Counter

import numpy as np
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

## 1. Two ways to summarise

### Intuition first
Give two students an article and ask each for a summary.

The first takes a **highlighter** and marks the three most important sentences. The summary consists of those sentences, word for word. This is **extractive** summarisation.

The second reads the article, puts it down, and **writes** a few sentences in their own words. This is **abstractive** summarisation.

| | Extractive | Abstractive |
|---|---|---|
| How | selects sentences from the source | generates new sentences |
| Can it state something false? | no: every sentence is from the source | **yes** |
| Fluency | sentences can read awkwardly out of context | usually smooth and well connected |
| Compression | limited to whole sentences | can merge and shorten freely |
| Needs | counting and a little linear algebra | a large trained language model |

The second row is the one to remember. An extractive summary may be badly chosen, but it is made of sentences that really were in the document. An abstractive summary is produced by a decoder, and you saw in Notebook 13 that a decoder writes fluent output whether or not the content is right.

### Our text
We need a text to work on and, for evaluation, a summary written by a person. Here is a short feature article about an invented town, and a reference summary of three sentences.

In [ ]:
article = """On a grey Tuesday morning, a queue of schoolchildren stretched around the corner of Harbour Street in Marlow Bay. They were waiting for the doors of the new town library to open for the first time. The old library closed four years ago after a storm flooded its basement and destroyed a third of its books. Since then, residents have had to travel twenty kilometres to the nearest branch in Eastfield. The new library cost 3.2 million euros and took two years to build. Most of the money came from the regional government, and the rest was raised by residents through concerts, bake sales and a sponsored swim across the bay. The building holds 40,000 books, twice as many as the old one. It also has a room with thirty computers, a recording studio and a workshop with 3D printers. The architect, Elena Voss, designed the building to stand on concrete pillars so that flood water can pass underneath it. Solar panels on the roof provide about half of the electricity the library needs. The library will open seven days a week, and it will stay open until ten in the evening on weekdays. Head librarian Tomas Reyes said that the long opening hours matter most to people who work during the day. He expects about 500 visitors a day in the first month. Not everyone in the town welcomed the project. Some residents argued that the money should have been spent on repairing the sea wall, which was also damaged in the storm. The town council replied that the sea wall repairs are funded separately and will begin next spring. A survey carried out last year found that 68 percent of residents supported the new library. The library also plans to lend more than books. From next month, members will be able to borrow tools, musical instruments and even telescopes. Evening classes in coding, languages and local history start in September. For the children at the front of the queue, none of that mattered very much. They wanted to know only one thing: where the comics were."""

reference = ("Marlow Bay has opened a new library, four years after a storm flooded the old one. "
             "The 3.2 million euro building holds 40,000 books and is raised on pillars to protect it from floods. "
             "It will open seven days a week, and although some residents wanted the money spent on the sea wall, most supported the project.")

sentences = nltk.sent_tokenize(article)
print(len(sentences), "sentences,", len(article.split()), "words")
print("Reference summary:", len(reference.split()), "words")

## 2. A baseline: the first three sentences

Before anything clever, the simplest method there is: take the **first three sentences**. It is known as *lead-3*.

In [ ]:
def show(summary_sentences):
    for sentence in summary_sentences:
        print(" -", sentence)

lead = sentences[:3]
show(lead)

For news reports, lead-3 is famously hard to beat, because journalists are trained to put the essential facts in the opening paragraph. Our article is a feature. It opens with a scene, and the facts come later. A method that works because of a convention fails when the convention is not followed. Still, it is our baseline, and anything we build must do better.

## 3. Extractive summarisation by word frequency

### Intuition first
What makes a sentence important? One of the oldest ideas, due to Hans Peter Luhn in 1958, is this: **the words that a text uses often are what the text is about, and a sentence that contains many of them is a central sentence.**

So: count the content words of the document, score each sentence by the frequencies of the words in it, and keep the best.

In [ ]:
def content_words(text):
    return [word for word in re.findall(r"[a-z]+", text.lower()) if word not in stop_words and len(word) > 2]

frequencies = Counter(content_words(article))
print(frequencies.most_common(10))

The most frequent content words point straight at the subject: the *library*, the *residents*, the *town*, its *books*. Now the sentence scores. The score of a sentence is simply the sum of the frequencies of its content words.

In [ ]:
def frequency_scores(sentences):
    frequencies = Counter(content_words(" ".join(sentences)))
    scores = []
    for sentence in sentences:
        scores.append(sum(frequencies[word] for word in content_words(sentence)))
    return scores

def pick(sentences, scores, how_many=3):
    best = sorted(range(len(sentences)), key=lambda i: scores[i], reverse=True)[:how_many]
    return [sentences[i] for i in sorted(best)]              # keep the original order

by_frequency = pick(sentences, frequency_scores(sentences))
show(by_frequency)

Note the last line of `pick`: the chosen sentences are put back in their original order, so that the summary reads in the order of the article.

The opening scene has gone, and sentences carrying facts have taken its place. The method has two weaknesses. Because scores are sums, long sentences are favoured over short ones. And a word such as *library* occurs in many sentences, so sentences are rewarded for containing it whether or not they add anything new. The next method looks at how sentences relate to **each other**.

## 4. TextRank: sentences that vote for each other

### Intuition first
In 1998 the founders of Google ranked web pages with a simple idea, **PageRank**: a page is important if important pages link to it. Each link is a vote, and a vote from an important page counts for more.

**TextRank** (Mihalcea and Tarau, 2004) applies the same idea to sentences. There are no links between sentences, so **similarity** takes their place. A sentence that is similar to many other sentences expresses something the document keeps returning to. Each sentence "votes" for the sentences it resembles, and votes from central sentences count for more.

### Step 1: the similarity graph
We represent each sentence as a TF-IDF vector and compute the cosine similarity of every pair, exactly as in Notebook 08. The result is a table with one row and one column per sentence, which we can read as a **graph**: sentences are the nodes, and the similarity between two of them is the weight of the edge joining them.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def similarity_matrix(sentences):
    vectors = TfidfVectorizer(stop_words="english").fit_transform(sentences)
    similarity = (vectors @ vectors.T).toarray()
    np.fill_diagonal(similarity, 0)                          # a sentence does not vote for itself
    return similarity

similarity = similarity_matrix(sentences)
print("Shape:", similarity.shape)

i, j = np.unravel_index(similarity.argmax(), similarity.shape)
print(f"\nMost similar pair (similarity {similarity[i, j]:.2f}):")
print(" -", sentences[i])
print(" -", sentences[j])

### Step 2: PageRank by hand

### Formal definition
Let $w_{ji}$ be the similarity between sentences $j$ and $i$. The score of sentence $i$ is defined in terms of the scores of all the others:

$$S(i) = \frac{1 - d}{N} + d \sum_{j \ne i} \frac{w_{ji}}{\sum_k w_{jk}} \, S(j)$$

Each sentence $j$ shares out its own score among the sentences it is similar to, in proportion to the similarities. The constant $d$, the **damping factor**, is usually 0.85. The remaining $1 - d$ is spread equally, so that no sentence ends up with nothing.

The definition is circular: every score depends on the others. We solve it by **iteration**. Start with equal scores, apply the formula to get new scores, and repeat. The scores settle down after a few dozen rounds.

In [ ]:
def pagerank(similarity, damping=0.85, iterations=50, report=False):
    n = len(similarity)
    row_sums = similarity.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1                              # a sentence similar to nothing gives no votes
    shares = similarity / row_sums                           # how each sentence divides its vote
    scores = np.ones(n) / n
    for step in range(iterations):
        new_scores = (1 - damping) / n + damping * shares.T @ scores
        change = np.abs(new_scores - scores).sum()
        scores = new_scores
        if report and step in (0, 1, 5, 20, 49):
            print(f"round {step + 1:>2}: total change {change:.6f}")
    return scores

textrank_scores = pagerank(similarity, report=True)

The change from one round to the next shrinks rapidly towards zero: the scores have converged. We can check our implementation against the `networkx` library.

In [ ]:
import networkx as nx

graph = nx.from_numpy_array(similarity)
library_scores = nx.pagerank(graph, alpha=0.85, weight="weight")

ours = np.argsort(-textrank_scores)[:5].tolist()
theirs = sorted(library_scores, key=library_scores.get, reverse=True)[:5]
print("Our top five sentences:     ", ours)
print("networkx's top five:        ", theirs)

In [ ]:
by_textrank = pick(sentences, textrank_scores)
show(by_textrank)

Three sentences chosen with no training, no labelled data and no knowledge of libraries or of English beyond a stop-word list. And because every sentence is copied from the article, nothing in this summary can be false to the source.

Now we have three candidate summaries. Which is best? Reading them gives an impression. We want a number.

## 5. Measuring summaries: ROUGE

### Intuition first
In Notebook 13 we scored translations with BLEU, by comparing them with a human reference. Summaries are scored in the same spirit, with one change of emphasis. BLEU is built on **precision**: how much of what the system wrote is in the reference? For a summary the first worry is the opposite: how much of what **should** be there did the system include? That is **recall**, and it gives the measure its name: **ROUGE**, *Recall-Oriented Understudy for Gisting Evaluation*.

### Formal definition
For n-grams of a chosen size $n$, count those that the candidate and the reference have in common (with clipping, as in BLEU). Then

$$\text{recall} = \frac{\text{n-grams in common}}{\text{n-grams in the reference}} \qquad \text{precision} = \frac{\text{n-grams in common}}{\text{n-grams in the candidate}} \qquad F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

- **ROUGE-1** uses single words: is the right content there?
- **ROUGE-2** uses pairs of words: is it expressed in similar phrases?
- **ROUGE-L** uses the **longest common subsequence**: the longest sequence of words that appear in both texts in the same order, not necessarily next to one another. It rewards keeping the order of the reference without demanding identical phrases.

In [ ]:
def tokens_of(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def f1(precision, recall):
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def rouge_n(candidate, reference, n):
    candidate_tokens, reference_tokens = tokens_of(candidate), tokens_of(reference)
    candidate_ngrams = Counter(tuple(candidate_tokens[i:i + n]) for i in range(len(candidate_tokens) - n + 1))
    reference_ngrams = Counter(tuple(reference_tokens[i:i + n]) for i in range(len(reference_tokens) - n + 1))
    common = sum((candidate_ngrams & reference_ngrams).values())          # & keeps the smaller count: clipping
    recall = common / max(1, sum(reference_ngrams.values()))
    precision = common / max(1, sum(candidate_ngrams.values()))
    return recall, precision, f1(precision, recall)

print("ROUGE-1 (recall, precision, F1):", [round(value, 2) for value in rouge_n("the cat sat on the mat", "the cat lay on the mat", 1)])
print("ROUGE-2 (recall, precision, F1):", [round(value, 2) for value in rouge_n("the cat sat on the mat", "the cat lay on the mat", 2)])

Five of the six words agree, and three of the five word pairs. For ROUGE-L we need the longest common subsequence, which is computed with a table, in the style of the dynamic programming you met in Notebooks 05 and 06: the answer for two texts is built from the answers for their beginnings.

In [ ]:
def lcs_length(a, b):
    table = [[0] * (len(b) + 1) for row in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1        # the words match: extend the subsequence
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[len(a)][len(b)]

def rouge_l(candidate, reference):
    candidate_tokens, reference_tokens = tokens_of(candidate), tokens_of(reference)
    longest = lcs_length(candidate_tokens, reference_tokens)
    recall = longest / max(1, len(reference_tokens))
    precision = longest / max(1, len(candidate_tokens))
    return recall, precision, f1(precision, recall)

print("LCS length:", lcs_length(tokens_of("the cat sat on the mat"), tokens_of("on the mat the cat sat")))
print("ROUGE-L:", [round(value, 2) for value in rouge_l("the cat sat on the mat", "the cat lay on the mat")])

### The comparison
Now all the methods against the reference, with a summary of three **random** sentences as a sanity check. For the random method we average over many draws, since any single draw could be lucky.

In [ ]:
def report(name, summary_text):
    r1 = rouge_n(summary_text, reference, 1)[2]
    r2 = rouge_n(summary_text, reference, 2)[2]
    rl = rouge_l(summary_text, reference)[2]
    print(f"{name:<22} {r1:>8.2f} {r2:>8.2f} {rl:>8.2f} {len(summary_text.split()):>7}")
    return r1, r2, rl

random.seed(0)
random_scores = np.mean([[rouge_n(" ".join(random.sample(sentences, 3)), reference, 1)[2],
                          rouge_n(" ".join(random.sample(sentences, 3)), reference, 2)[2],
                          rouge_l(" ".join(random.sample(sentences, 3)), reference)[2]] for i in range(200)], axis=0)

print(f"{'method (F1 scores)':<22} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8} {'words':>7}")
print(f"{'random (average)':<22} {random_scores[0]:>8.2f} {random_scores[1]:>8.2f} {random_scores[2]:>8.2f}")
results = {
    "lead-3": report("lead-3", " ".join(lead)),
    "word frequency": report("word frequency", " ".join(by_frequency)),
    "TextRank": report("TextRank", " ".join(by_textrank)),
}

Read the table as you would any experiment in this course. The random row tells us what three sentences picked by chance would score. On this article the oldest and simplest method, word frequency, does best. TextRank does no better than the first three sentences, and neither is far above chance.

Notice also how low all the numbers are in absolute terms. Our reference is itself **abstractive**: its author merged facts from several sentences into one and chose different words. No selection of three original sentences can match it closely. ROUGE scores are not percentages of correctness. They only have meaning in comparison, between systems, on the same documents, with the same references.

### One article is an anecdote
Should we conclude that word frequency beats TextRank? Not from a single text. With one article and one reference, the difference between two methods may be an accident of which sentences happened to share words with the reference.

A proper evaluation needs many documents, each with a reference. We have such a collection at hand. Every article in the Reuters corpus of Notebook 08 begins with a **headline**, which is a one-line summary written by an editor. So for 500 articles of at least eight sentences, let each method choose **one** sentence, and score it against the headline.

In [ ]:
from nltk.corpus import reuters

def headline_evaluation(shuffle=False, how_many=500):
    """Average ROUGE-1 F1 of one chosen sentence against the headline, for each method."""
    random.seed(0)
    scores = {"first sentence": [], "word frequency": [], "TextRank": [], "random sentence": []}
    for file_id in reuters.fileids():
        raw = reuters.raw(file_id)
        if "\n" not in raw:
            continue
        headline, body = raw.split("\n", 1)
        parts = [" ".join(sentence.split()) for sentence in nltk.sent_tokenize(body)]
        if len(parts) < 8 or len(headline.split()) < 4:
            continue
        if shuffle:
            random.shuffle(parts)                    # same sentences, in a random order

        candidates = {
            "first sentence": parts[0],
            "word frequency": parts[int(np.argmax(frequency_scores(parts)))],
            "TextRank": parts[int(np.argmax(pagerank(similarity_matrix(parts))))],
            "random sentence": random.choice(parts),
        }
        for name, sentence in candidates.items():
            scores[name].append(rouge_n(sentence, headline, 1)[2])
        if len(scores["TextRank"]) == how_many:
            break
    return {name: float(np.mean(values)) for name, values in scores.items()}

in_order = headline_evaluation()

print("Average ROUGE-1 F1 against the headline, over 500 articles\n")
for name, value in sorted(in_order.items(), key=lambda item: -item[1]):
    print(f"{name:<18} {value:.3f}")

Over 500 articles the picture is clear, and it differs from the single article.

- Both of our methods are well above a random sentence. They do find central sentences.
- **TextRank** is somewhat ahead of word frequency.
- The **first sentence** beats them both by a wide margin.

The last point is the lesson of Section 2, now with evidence behind it. These are news reports, and news is written with the essential facts first. A method that knows nothing but "take the beginning" exploits that convention fully. It is the reason every serious paper on news summarisation reports the lead baseline, and the reason a new method must be tested on other kinds of text too, where the convention does not hold.

### When does each method win?
We can test that last claim directly, with a controlled experiment. Take the same 500 articles and put the sentences of each one in a **random order**. Every sentence is still there, and the headline is the same. The only thing removed is the convention that the important sentence comes first. This imitates the many kinds of text that have no such convention: a transcript of a meeting, a thread of messages, a set of notes.

In [ ]:
shuffled = headline_evaluation(shuffle=True)

print(f"{'':<18} {'news order':>11} {'random order':>13}")
for name in ["first sentence", "TextRank", "word frequency", "random sentence"]:
    print(f"{name:<18} {in_order[name]:>11.3f} {shuffled[name]:>13.3f}")

Read the two columns side by side.

- The **first sentence** collapses to the level of a random sentence. It never knew anything about the content. All of its success came from the way journalists arrange their text.
- **TextRank** and **word frequency** are unchanged. They look only at what the sentences say, so the order makes no difference to them. In the second column TextRank is the best method.

| | Text with the key facts first (news) | Text with no fixed order |
|---|---|---|
| What the first-sentence baseline relies on | a convention of the genre | nothing: the convention is absent |
| What TextRank relies on | which sentences are central to the content | the same |
| Better method | **first sentence** | **TextRank** |

Neither method is better in general. The simple one wins when the structure of the text does its work for it, and the content-based one wins when there is no such structure to lean on. When you meet a claim that a new method "beats the baseline", the first thing to ask is: **on what kind of data?**

> ⚠️ **Common pitfalls**
>
> - Drawing conclusions from one document. A proper evaluation averages over hundreds of documents. On a single article the ranking of two methods can easily be an accident.
> - Comparing ROUGE scores from different papers or tools. Stemming, stop-word removal and tokenisation all change the numbers.
> - Forgetting that ROUGE, like BLEU, **counts words and knows nothing of truth**. A summary that says the library cost *32 million* differs from the correct one by a single token.

## 6. Abstractive summarisation with a pretrained model

*This section downloads a model of about 1.2 GB and is meant for Colab.*

An abstractive summariser is a sequence-to-sequence model, as in Notebook 13: an encoder reads the article and a decoder writes the summary. The one below is a Transformer trained on several hundred thousand news articles paired with summaries written by journalists.

In [ ]:
from transformers import pipeline

summariser = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
abstractive = summariser(article, max_length=90, min_length=40, do_sample=False)[0]["summary_text"].strip()

print(abstractive, "\n")
print(f"{'method (F1 scores)':<22} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8} {'words':>7}")
results["abstractive"] = report("abstractive model", abstractive)

Read the model's summary slowly, and check **every statement** against the article.

- Is it fluent? Almost certainly, and more smoothly connected than our extracted sentences.
- Did it rewrite, or mostly copy? Models trained on news tend to copy long stretches, especially from the beginning of the text.
- Is everything in it **true to the article**? Look hard at the numbers, the names, and who said or did what.

A statement in a summary that the source does not support is called a **hallucination**. It is the characteristic failure of abstractive systems, and it is dangerous for a simple reason: the false statement is written in the same fluent, confident prose as the true ones.

### A simple check for unsupported claims
Complete verification needs a human reader. Some errors can be caught mechanically, though. Numbers and names are where summaries most often go wrong, and both are easy to find with the regular expressions of Notebook 02. The function below lists every number and every capitalised name in a summary that does **not** occur in the source.

In [ ]:
def unsupported_items(summary, source):
    numbers = re.findall(r"\d+(?:[.,]\d+)?", summary)
    names = re.findall(r"(?<![.!?] )(?<!^)\b[A-Z][a-z]+\b", summary)       # capitalised words not at the start of a sentence
    return [item for item in numbers + names if item not in source]

faithful = "The new library in Marlow Bay cost 3.2 million euros and holds 40,000 books."
invented = "The new library in Marlow Bay cost 5.4 million euros and was designed by Elena Voss and Martin Hale."

print("Faithful summary:", unsupported_items(faithful, article))
print("Invented summary:", unsupported_items(invented, article))

The check lets a correct summary through and catches the wrong price and the invented architect. It is a safety net with large holes: it cannot detect that two real names have been swapped, or that a true number has been attached to the wrong thing. It costs nothing, though, and it is the kind of guard that any system producing summaries for people to rely on should have.

In [ ]:
print("Unsupported items in the model's summary:", unsupported_items(abstractive, article))

> 🧠 **Which kind should you use?** If every statement must be traceable to the source, as in law, medicine or compliance, an **extractive** summary, or an abstractive one that is shown alongside its sources and checked, is the responsible choice. If readability matters most and a person reviews the result, **abstractive** models give far more natural summaries. The best current systems combine the two: retrieve or select the relevant passages first, and then write from them. That combination is the subject of the final notebook.

---
## ✏️ Exercises

### Exercise 1 (ROUGE on paper)
For the reference `"the library opened after four years"` and the candidate `"the new library opened on tuesday"`, compute by hand the ROUGE-1 recall, precision and F1, and the length of the longest common subsequence. Check with the functions above.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
reference_1 = "the library opened after four years"
candidate_1 = "the new library opened on tuesday"

# words in common: the, library, opened -> 3
# recall = 3/6 = 0.50      precision = 3/6 = 0.50      F1 = 0.50
# longest common subsequence: "the library opened" -> 3
print([round(value, 2) for value in rouge_n(candidate_1, reference_1, 1)])
print(lcs_length(tokens_of(candidate_1), tokens_of(reference_1)))
```

*"the library opened" is a common subsequence although "new" stands between "the" and "library" in the candidate. A subsequence keeps the order and may skip words, which is what distinguishes ROUGE-L from ROUGE-2.*
</details>

### Exercise 2 (Longer summaries)
Using TextRank, build summaries of 1, 2, 3, 5 and 8 sentences. For each, print the ROUGE-1 **recall** and **precision** separately. What happens to each as the summary grows, and why is recall alone a poor measure?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print(f"{'sentences':>9} {'words':>6} {'recall':>7} {'precision':>10} {'F1':>6}")
for how_many in [1, 2, 3, 5, 8]:
    summary = " ".join(pick(sentences, textrank_scores, how_many))
    recall, precision, score = rouge_n(summary, reference, 1)
    print(f"{how_many:>9} {len(summary.split()):>6} {recall:>7.2f} {precision:>10.2f} {score:>6.2f}")
```

*Recall can only rise as sentences are added, and it would reach its maximum if we returned the whole article. Precision falls, because more and more of the summary is absent from the reference. A summary is supposed to be short, so recall must be balanced against precision, or the length must be fixed in advance, as it is in shared evaluations.*
</details>

### Exercise 3 (Position as a clue)
In many kinds of text, important sentences come early. Write `position_scores(sentences)` that gives the first sentence a score of 1, falling steadily to 0 for the last. Then build a combined score: the TextRank score divided by its maximum, plus 0.5 times the position score. Does the combined summary score better on ROUGE than TextRank alone, for this article?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def position_scores(sentences):
    n = len(sentences)
    return np.array([1 - i / (n - 1) for i in range(n)])

combined = textrank_scores / textrank_scores.max() + 0.5 * position_scores(sentences)
by_combination = pick(sentences, combined)
show(by_combination)

print(f"\n{'method (F1 scores)':<22} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8} {'words':>7}")
report("TextRank", " ".join(by_textrank))
report("TextRank + position", " ".join(by_combination))
```

*Whether position helps depends on how the text is organised. This article opens with a scene and puts its facts in the next few sentences, so a moderate preference for early sentences can help or hurt depending on the weight. A setting tuned on one article proves nothing: weights like this must be chosen on a set of development documents and tested on others.*
</details>

### Exercise 4 (Avoid repeating yourself)
A summary should not say the same thing twice, yet the highest-ranked sentences are often similar to one another. Write `pick_diverse(sentences, scores, similarity, how_many, limit)` that goes through the sentences from the best score down, and accepts a sentence only if its similarity to every sentence already accepted is below `limit`. Compare its choice with plain `pick` for 4 sentences and a limit of 0.15.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def pick_diverse(sentences, scores, similarity, how_many=3, limit=0.15):
    chosen = []
    for i in sorted(range(len(sentences)), key=lambda i: scores[i], reverse=True):
        if all(similarity[i, j] < limit for j in chosen):
            chosen.append(i)
        if len(chosen) == how_many:
            break
    return [sentences[i] for i in sorted(chosen)]

print("Plain selection:")
show(pick(sentences, textrank_scores, 4))
print("\nDiverse selection:")
show(pick_diverse(sentences, textrank_scores, similarity, 4, 0.15))
```

*TextRank rewards sentences that resemble many others, so its favourites tend to resemble each other. Refusing a sentence that is too close to one already chosen trades a little centrality for coverage of more of the article. This idea is known as maximal marginal relevance, and it returns in the next notebook when we select passages for a language model to read.*
</details>

### Exercise 5 (Summarise real news, a little harder)
Write a function `summarise(text, how_many=3)` that splits a text into sentences and returns a TextRank summary. Apply it to the longest article of the Reuters corpus under the category `coffee`. Print the article's headline (its first line) and your summary. Does the summary agree with the headline?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from nltk.corpus import reuters


def summarise(text, how_many=3):
    parts = [" ".join(sentence.split()) for sentence in nltk.sent_tokenize(text)]
    if len(parts) <= how_many:
        return parts
    return pick(parts, pagerank(similarity_matrix(parts)), how_many)

longest = max(reuters.fileids("coffee"), key=lambda file_id: len(reuters.raw(file_id)))
raw = reuters.raw(longest)
headline, body = raw.split("\n", 1)

print("HEADLINE:", headline.strip())
print(f"({len(nltk.sent_tokenize(body))} sentences)\n")
show(summarise(body))
```

*A headline is the shortest summary of all, written by an editor. If the extracted sentences are about what the headline announces, the method has found the centre of the article. Using headlines as free reference summaries is exactly how the large training sets for summarisation models were built.*
</details>

### Exercise 6 (Catch the errors)
The summary below contains three statements that the article does not support. First find them by reading. Then see which of them `unsupported_items` catches, and explain why it misses the others.

In [ ]:
suspect = ("The new library in Marlow Bay cost 3.2 million euros and holds 80,000 books. "
           "Head librarian Elena Voss expects 500 visitors a day. "
           "A survey found that 68 percent of residents opposed the project.")
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print(unsupported_items(suspect, article))
```

*The article says 40,000 books, so "80,000" is caught: that number appears nowhere in the source. The other two errors pass unnoticed. Elena Voss is the architect, not the librarian, but her name does occur in the article. And 68 percent of residents supported the project, they did not oppose it, but every word of the false sentence can be found in the source. Checking that the pieces exist is far easier than checking that they are put together correctly. That gap is why summaries that matter still need a human reader.*
</details>

## 🔑 Key takeaways

- **Extractive** summarisation selects sentences from the source and cannot invent. **Abstractive** summarisation writes new text, reads better, and can state things the source never said.
- Always start from a **baseline**. For news, the first sentences are hard to beat, because of how news is written. On text with no such convention, a content-based method such as TextRank wins.
- **Word frequency** scores a sentence by how many of the document's frequent words it contains.
- **TextRank** builds a graph of sentence similarities and ranks it with **PageRank**: a sentence is central if it resembles other central sentences.
- **ROUGE** compares a summary with a human reference: ROUGE-1 on words, ROUGE-2 on word pairs, ROUGE-L on the longest common subsequence. Report recall, precision and F1.
- ROUGE is meaningful only in comparison and over many documents, and it is **blind to truth**.
- **Hallucination** is the main risk of abstractive systems. Simple checks catch some of it. Careful reading catches the rest.

---
**Next:** *Notebook 18: Semantic Search and Retrieval-Augmented Generation*, the final notebook, where the search of Notebook 08, the embeddings of Notebook 10 and the language models of this module come together in one system.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*